# BigQuery Timestamp Fetch (REST API)
Reads `data/input/seeds.csv` and retrieves block timestamps using the BigQuery REST API.

In [ ]:
import pandas as pd
import requests
from google.auth import default
from google.auth.transport.requests import Request

seeds = pd.read_csv('../data/input/seeds.csv', header=None)[0].tolist()
credentials, project_id = default(scopes=["https://www.googleapis.com/auth/bigquery"])
credentials.refresh(Request())
access_token = credentials.token
sql = """
SELECT TO_HEX(`hash`) AS txid, block_timestamp
FROM `bigquery-public-data.crypto_bitcoin.transactions`
WHERE TO_HEX(`hash`) IN UNNEST(@txids)
"""
body = {
  "query": sql,
  "useLegacySql": False,
  "parameterMode": "NAMED",
  "queryParameters": [
    {"name": "txids", "parameterType": {"type": "ARRAY", "arrayType": {"type": "STRING"}},
     "parameterValue": {"arrayValues": [{"value": t} for t in seeds]}}
  ]
}
response = requests.post(
    f"https://bigquery.googleapis.com/bigquery/v2/projects/{project_id}/queries",
    headers={"Authorization": f"Bearer {access_token}"},
    json=body,
)
rows = response.json().get("rows", [])
timestamps = {r['f'][0]['v']: r['f'][1]['v'] for r in rows}
pd.DataFrame(list(timestamps.items()), columns=['txid','timestamp']).head()